### Final Project CSE 476
#### Group members: Neil Shenoy, Kavish Sharma, Jenna Skabelund

In [1]:
%pip install requests python-dotenv

import os, json, textwrap, re, time
import requests

API_KEY = "sk-THSHSCeQUCzILV98j-3xCQ"
API_BASE = os.getenv("API_BASE", "https://openai.rc.asu.edu/v1")  
MODEL    = os.getenv("MODEL_NAME", "qwen3-30b-a3b-instruct-2507")              

def call_model_chat_completions(prompt: str,
                                system: str = "You are a helpful assistant. Reply with only the final answer—no explanation.",
                                model: str = MODEL,
                                temperature: float = 0.0,
                                max_tokens: int = 128,
                                timeout: int = 60) -> dict:
    """
    Calls an OpenAI-style /v1/chat/completions endpoint and returns:
    { 'ok': bool, 'text': str or None, 'raw': dict or None, 'status': int, 'error': str or None, 'headers': dict }
    """
    url = f"{API_BASE}/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
    }
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    try:
        resp = requests.post(url, headers=headers, json=payload, timeout=timeout)
        status = resp.status_code
        hdrs   = dict(resp.headers)
        if status == 200:
            data = resp.json()
            text = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            return {"ok": True, "text": text, "raw": data, "status": status, "error": None, "headers": hdrs}
        else:
            # try best-effort to surface error text
            err_text = None
            try:
                err_text = resp.json()
            except Exception:
                err_text = resp.text
            return {"ok": False, "text": None, "raw": None, "status": status, "error": str(err_text), "headers": hdrs}
    except requests.RequestException as e:
        return {"ok": False, "text": None, "raw": None, "status": -1, "error": str(e), "headers": {}}


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Chain of thought prompting
def CotPrompting(inp: str):
    CotResponse = call_model_chat_completions(
        inp, 
        "You are a helpful assistant. Answer the question while thinking step by step. Return only the final answer.",
        max_tokens=512
    )
    text = (CotResponse.get("text") or "").strip()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1] if lines else text

# test
x = CotPrompting("What is 17 * 26? Do not use latex symbols")
print(x)

So, 17 * 26 = 442.


In [3]:
# Self consistency
def SelfConsistency(inp):
    resp1 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()
    resp2 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()
    resp3 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()
    resp4 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()
    resp5 = call_model_chat_completions(inp, temperature=0.7)["text"].strip()

    finalResp = call_model_chat_completions(
        f"Here are the 5 responses I got when calling you earlier for question {inp}, which is the most consistent? \
            {resp1}, {resp2}, {resp3}, {resp4}, {resp5}",
        "You are a helpful LLM. Reply with only the most common answer, nothing else."
    )

    return finalResp["text"].strip()

# test
y = SelfConsistency("What is 123 * 412?")
print(y)

50676


In [4]:
# Best Of N
def BestOfN(inp: str, n: int):
    resp = []
    for i in range(n):
        resp.append(call_model_chat_completions(inp)["text"].strip()) 
    
    finalResp = call_model_chat_completions(
        f"Here are the {n} responses I got when calling you earlier for question {inp}. Which is the best answer? {resp}",
        "You are a helpful LLM that will give me the best answer. Return only the final answer."
    )
    
    return finalResp["text"].strip()

# test
print(BestOfN("What is the capital of Syria?", 3))

Damascus


In [5]:
#Helper
def get_response_text(response_dict):
    if not isinstance(response_dict, dict):
        return ""
    return (response_dict.get("text") or "").strip()

In [6]:
#ask for an answer, then ask the model "is this answer correct? if not, fix it." Two API calls total.

def solve_with_reflection(question, model=MODEL, temperature=0.0, too_wordy=False):

    first_system = (
        "You are great at finding the correct answers to any given questions. "
        "Solve the user's question. Return only the final answer. Do not explain."
    )

    first_prompt = f"""Question: {question} What is the answer?"""
    first_response = call_model_chat_completions(
        first_prompt,
        system=first_system,
        model=model,
        temperature=temperature,
    )
    initial_answer = get_response_text(first_response)


    reflection_system = (
        "You are great at reviewing questions. "
        "Check whether the proposed answer is correct and complete. "
        "If it is wrong, fix it. "
        "Return only the correct and complete final answer."
    )

    reflection_prompt = f"""You are reviewing a previous answer. Original question: {question} Previous answer: {initial_answer} Task: Check the answer carefully. If it is correct, keep it. If it is wrong or incomplete, fix it. Return the corrected final answer only."""

    reflection_response = call_model_chat_completions(
        reflection_prompt,
        system=reflection_system,
        model=model,
        temperature=0.0,
    )
    
    reflected_answer = get_response_text(reflection_response)
    final_answer = reflected_answer if reflected_answer else initial_answer

    if too_wordy:
        print("QUESTION:", question)
        print("INITIAL ANSWER:", initial_answer)
        print("NEW REFLECTED ANSWER:", reflected_answer)
        print("FINAL ANSWER:", final_answer)

    return {
        "initial_answer": initial_answer,
        "reflected_answer": reflected_answer,
        "final_answer": final_answer,
    }

In [7]:
#Test

example_question = "What is 17 * 26?"
reflection_result = solve_with_reflection(example_question, too_wordy=True)

QUESTION: What is 17 * 26?
INITIAL ANSWER: 442
NEW REFLECTED ANSWER: 442
FINAL ANSWER: 442


In [8]:
#ask the model to break the question into smaller sub-questions, answer each one, then combine them into a final answer

def solve_with_step_decomposition(question, model=MODEL, temperature=0.0, too_wordy=False):

    decomposition_system = (
        "You are good at breaking problems into smaller steps. "
        "Break the user's question into clear sub-questions or steps. "
        "Do not solve the problem yet."
    )

    decomposition_prompt = f"""Question:{question} Break this question into smaller steps needed to solve it."""

    decomposition_response = call_model_chat_completions(
        decomposition_prompt,
        system=decomposition_system,
        model=model,
        temperature=temperature,
    )

    steps = get_response_text(decomposition_response)


    solve_system = (
        "You are a careful problem solver who always gets the correct question by breaking it down. "
        "Use the provided steps to solve the question. "
        "Return only the final answer."
    )

    solve_prompt = f"""Original question:{question} Steps to follow: {steps} Now solve the original question. Return only the final answer."""

    solve_response = call_model_chat_completions(
        solve_prompt,
        system=solve_system,
        model=model,
        temperature=temperature,
    )

    final_answer = get_response_text(solve_response)


    if too_wordy:
        print("QUESTION:", question)
        print("DECOMPOSED STEPS:", steps)
        print("FINAL ANSWER:", final_answer)

    return {
        "steps": steps,
        "final_answer": final_answer,
    }

In [9]:
#Test

example_question = "How many minutes are in a year?"
step_result = solve_with_step_decomposition(example_question, too_wordy=True)

QUESTION: How many minutes are in a year?
DECOMPOSED STEPS: Sure! Let's break down the question "How many minutes are in a year?" into smaller, manageable steps:

1. **Determine the number of days in a year**  
   - Is it a common year (365 days) or a leap year (366 days)?  
   - We’ll typically assume a common year unless specified.

2. **Find the number of hours in a day**  
   - There are 24 hours in a day.

3. **Calculate the number of minutes in an hour**  
   - There are 60 minutes in an hour.

4. **Calculate the
FINAL ANSWER: 365 days × 24 hours/day × 60 minutes/hour = 525,600 minutes

Final answer: 525600


In [10]:
#Helper


def grader_check(question, proposed_answer, model=MODEL):
    grader_system = (
        "You are a grader that is very strict. "
        "Check whether the proposed answer correctly answers the question. "
        "Reply with only YES or NO."
    )

    grader_prompt = f"""Question:{question} Proposed answer: {proposed_answer} Is the proposed answer correct? Reply only YES or NO."""

    grader_response = call_model_chat_completions(
        grader_prompt,
        system=grader_system,
        model=model,
        temperature=0.0,
    )

    grader_text = get_response_text(grader_response).upper()

    return grader_text.startswith("YES")

In [11]:
# if the model's answer fails your grader check, send it back with "that was wrong, try again" and loop until it gets it right or hits a max attempt limit.


def solve_with_iterative_refinement(question, model=MODEL, temperature=0.0, max_attempts=3, too_wordy=False):

    current_answer = ""
    attempt_history = []

    for attempt in range(1, max_attempts + 1):

        if attempt == 1:
            prompt = f"""Question:{question} Solve the question. Return only the final answer."""

        else:
            prompt = f"""Question:{question} Your previous answer was:{current_answer} That answer was marked wrong. Try again carefully. Return only the final answer."""

        answer_response = call_model_chat_completions(
            prompt,
            system="You are great at finding the correct answer. Return only the final answer.",
            model=model,
            temperature=temperature,
        )

        current_answer = get_response_text(answer_response)

        is_correct = grader_check(question, current_answer, model=model)

        attempt_history.append({
            "attempt": attempt,
            "answer": current_answer,
            "correct": is_correct,
        })

        if too_wordy:
            print("ATTEMPT:", attempt)
            print("ANSWER:", current_answer)
            print("CORRECT:", is_correct)
            print()

        if is_correct:
            break

    return {
        "final_answer": current_answer,
        "attempts": attempt_history,
    }

In [12]:
#Test

example_question = "What is 17 * (26 + 3 * 5)?"
refinement_result = solve_with_iterative_refinement(example_question, max_attempts=3, too_wordy=True)

ATTEMPT: 1
ANSWER: 17 * (26 + 3 * 5)  
= 17 * (26 + 15)  
= 17 * 41  
= 697

697
CORRECT: True



In [13]:
# LLM-as-a-Judge Implementation

def llm_judge(question: str, prediction: str, expected_answer: str, model=MODEL) -> bool:
    
    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."

    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False"""

    response = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (response.get("text") or "").strip().lower()

    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: normalized string match if model gives unexpected output
    normalize = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return normalize(prediction) == normalize(expected_answer)

In [14]:
#LLM-as-a-Judge Test Cases

test_cases = [
    #True 
    {
        "question": "What is the capital of France?",
        "prediction": "Paris",
        "expected": "Paris",
        "label": "exact match → True"
    },
    #True 
    {
        "question": "What is 3 + 4?",
        "prediction": "The answer is 7.",
        "expected": "7",
        "label": "wordy but correct → True"
    },
    #False
    {
        "question": "What is the capital of France?",
        "prediction": "Berlin",
        "expected": "Paris",
        "label": "wrong answer → False"
    },
    #True
    {
        "question": "What is 17 * 26?",
        "prediction": "442",
        "expected": "442",
        "label": "math correct → True"
    },
    #False
    {
        "question": "What is 17 * 26?",
        "prediction": "452",
        "expected": "442",
        "label": "math wrong → False"
    },
    #True
    {
        "question": "Does ice float on water?",
        "prediction": "Yes, ice floats because it is less dense than liquid water.",
        "expected": "Yes",
        "label": "correct but verbose → True"
    },
]

print("LLM-as-a-Judge Test Results")

all_passed = True
for tc in test_cases:
    result = llm_judge(tc["question"], tc["prediction"], tc["expected"])
    expected_bool = "True" in tc["label"]
    passed = result == expected_bool
    all_passed = all_passed and passed
    status = "PASS" if passed else "FAIL"
    print(f"{status}  [{tc['label']}]")
    print(f"       got: {result}")

print("All tests passed" if all_passed else "Test(s) failed")

LLM-as-a-Judge Test Results
PASS  [exact match → True]
       got: True
PASS  [wordy but correct → True]
       got: True
PASS  [wrong answer → False]
       got: False
PASS  [math correct → True]
       got: True
PASS  [math wrong → False]
       got: False
PASS  [correct but verbose → True]
       got: True
All tests passed


In [15]:
# Tool Use Implementation (2-Step)

def solve_with_tool_use(question: str, model=MODEL) -> dict:

    #Step 1
    tool_decision_system = (
        "You are a reasoning assistant. Your job is to decide if a question "
        "requires arithmetic calculation. "
        "If yes, extract a single Python-evaluable math expression (ie. 17*26, (3+5)/2). "
        "If no, reply with NONE. "
        "Reply in exactly this format:\n"
        "USE_TOOL: <expression or NONE>"
    )

    tool_decision_response = call_model_chat_completions(
        question,
        system=tool_decision_system,
        model=model,
        temperature=0.0,
    )

    decision_text = (tool_decision_response.get("text") or "").strip()

    expression = None
    match = re.search(r"USE_TOOL:\s*(.+)", decision_text, re.IGNORECASE)
    if match:
        candidate = match.group(1).strip()
        if candidate.upper() != "NONE":
            expression = candidate

    #Step 2a - eval
    if expression:
        try:
            allowed = {"__builtins__": {}}
            import math
            allowed.update(vars(math))
            tool_result = eval(expression, allowed)
            return {
                "used_tool": True,
                "expression": expression,
                "final_answer": str(tool_result),
            }
        except Exception as e:
            pass

    #Step 2b - direct
    direct_response = call_model_chat_completions(
        question,
        system="You are a helpful assistant. Reply with only the final answer—no explanation.",
        model=model,
        temperature=0.0,
    )

    return {
        "used_tool": False,
        "expression": expression,
        "final_answer": (direct_response.get("text") or "").strip(),
    }

In [16]:
#Tool Use Tests

tool_tests = [
    #eval
    {
        "question": "What is 17 * 26?",
        "expected": "442",
        "label": "basic multiplication → use tool",
        "expect_tool": True,
    },
    #eval
    {
        "question": "What is 17 * (26 + 3 * 5)?",
        "expected": "697",
        "label": "nested arithmetic → use tool",
        "expect_tool": True,
    },
    #eval
    {
        "question": "What is 144 / 12?",
        "expected": "12",
        "label": "division → use tool",
        "expect_tool": True,
    },
    #direct
    {
        "question": "What is the capital of Japan?",
        "expected": "Tokyo",
        "label": "factual question → no tool",
        "expect_tool": False,
    },
    #direct
    {
        "question": "Which is heavier, a pound of feathers or a pound of gold?",
        "expected": "They weigh the same",
        "label": "common sense → no tool",
        "expect_tool": False,
    },
]

print("Tool Use Test Results")

for tc in tool_tests:
    result = solve_with_tool_use(tc["question"])
    answer_correct = llm_judge(tc["question"], result["final_answer"], tc["expected"])
    tool_correct = result["used_tool"] == tc["expect_tool"]

    answer_status = "Pass" if answer_correct else "Fail"
    tool_status   = "Pass" if tool_correct   else "Fail"

    print(f"{answer_status} Answer  {tool_status} Tool  [{tc['label']}]")
    print(f"        used_tool:  {result['used_tool']}")
    if result["expression"]:
        print(f"        expression: {result['expression']}")
    print(f"        answer:     {result['final_answer']}")
    print()

Tool Use Test Results
Pass Answer  Pass Tool  [basic multiplication → use tool]
        used_tool:  True
        expression: 17 * 26
        answer:     442

Pass Answer  Pass Tool  [nested arithmetic → use tool]
        used_tool:  True
        expression: 17 * (26 + 3 * 5)
        answer:     697

Pass Answer  Pass Tool  [division → use tool]
        used_tool:  True
        expression: 144 / 12
        answer:     12.0

Pass Answer  Pass Tool  [factual question → no tool]
        used_tool:  False
        answer:     Tokyo

Fail Answer  Pass Tool  [common sense → no tool]
        used_tool:  False
        answer:     A pound of feathers and a pound of gold weigh the same—both are one pound. However, gold is measured in troy pounds, which are slightly heavier than the avoirdupois pounds used for feathers. So, in terms of troy weight, a pound of gold is heavier than a pound of feathers measured in avoirdupois weight. But if both are measured in the same system, they weigh the same.



In [17]:
#Step 3: solve()
def solve(question: str, model=MODEL) -> str:

    #domain classification
    classify_system = (
        "You are a question classifier. "
        "Reply with exactly one word from this list: "
        "math, coding, planning, common_sense, future_prediction. "
        "No punctuation. No explanation."
    )
    classify_response = call_model_chat_completions(
        question,
        system=classify_system,
        model=model,
        temperature=0.0,
    )
    domain = (classify_response.get("text") or "").strip().lower()

    #sanitize
    valid_domains = {"math", "coding", "planning", "common_sense", "future_prediction"}
    if domain not in valid_domains:
        domain = "fallback"

    candidates = []

    #techniques selected based on domain

    if domain == "math":
        tool_result = solve_with_tool_use(question, model=model)
        candidates.append(tool_result["final_answer"])

        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        refinement_result = solve_with_iterative_refinement(
            question, model=model, max_attempts=3
        )
        candidates.append(refinement_result["final_answer"])

    elif domain == "coding":
        decomp_result = solve_with_step_decomposition(question, model=model)
        candidates.append(decomp_result["final_answer"])

        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        refinement_result = solve_with_iterative_refinement(
            question, model=model, max_attempts=3
        )
        candidates.append(refinement_result["final_answer"])

    elif domain == "planning":
        decomp_result = solve_with_step_decomposition(question, model=model)
        candidates.append(decomp_result["final_answer"])

        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        reflection_result = solve_with_reflection(question, model=model)
        candidates.append(reflection_result["final_answer"])

    elif domain == "common_sense":
        bon_answer = BestOfN(question, n=3)
        candidates.append(bon_answer)

        sc_answer = SelfConsistency(question)
        candidates.append(sc_answer)

    elif domain == "future_prediction":
        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        reflection_result = solve_with_reflection(question, model=model)
        candidates.append(reflection_result["final_answer"])

    else:
        cot_answer = CotPrompting(question)
        candidates.append(cot_answer)

        reflection_result = solve_with_reflection(question, model=model)
        candidates.append(reflection_result["final_answer"])

    #Delete duplicate candidates
    seen = set()
    unique_candidates = []
    for c in candidates:
        normalized = re.sub(r"\s+", " ", (c or "").strip().lower())
        if normalized and normalized not in seen:
            seen.add(normalized)
            unique_candidates.append(c)

    if len(unique_candidates) == 1:
        return unique_candidates[0]

    #LLM-as-a-Judge picks best candidate 
    judge_system = (
        "You are an expert judge selecting the best answer to a question. "
        "You must return only one integer: 1, 2, 3, etc. "
        "Do not return the answer text. Do not return Candidate or Option."
    )

    candidates_block = "\n".join(
        f"{i+1}. {c}" for i, c in enumerate(unique_candidates)
    )

    judge_prompt = f"""
    Question:
    {question}

    Candidate answers:
    {candidates_block}

    Which candidate is best?
    Return only the candidate number.
    """

    judge_response = call_model_chat_completions(
        judge_prompt,
        system=judge_system,
        model=model,
        temperature=0.0,
        max_tokens=10,
    )

    judge_text = (judge_response.get("text") or "").strip()
    m = re.search(r"\d+", judge_text)

    if m:
        idx = int(m.group(0)) - 1
        if 0 <= idx < len(unique_candidates):
            return extract_final_answer(unique_candidates[idx])

    return extract_final_answer(unique_candidates[0])

def extract_final_answer(text: str) -> str:
    """If the output looks like reasoning, try to pull the last number or short phrase."""
    if not text:
        return text
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    # If it's already short and clean, return as-is
    if len(text) < 50 and '\n' not in text.strip():
        return text
    # Look for common answer patterns
    for line in reversed(lines):
        if any(kw in line.lower() for kw in ["answer is", "answer:", "= ", "therefore", "thus"]):
            # Extract trailing number or word
            m = re.search(r'[\$]?([\d,]+\.?\d*)\s*$', line)
            if m:
                return m.group(1).replace(',', '')
    # Fallback: return last line
    return lines[-1] if lines else text

In [18]:
# import json

# data_file = "final_project_tutorial_and_dev_data/cse476_final_project_dev_data.json"

# with open(data_file, "r") as f:
#     test_data = json.load(f)

# results = []

# for i, item in enumerate(test_data):
#     question = item["input"]

#     print(f"Running question {i+1}/{len(test_data)}")

#     try:
#         model_answer = solve(question)
#     except Exception as e:
#         print("Error:", e)
#         model_answer = "ERROR"

#     results.append({
#         "question_number": i + 1,
#         "input": question,
#         "expected_output": item.get("output", ""),
#         "model_output": model_answer,
#         "domain": item.get("domain", "")
#     })

# print("Finished running all questions.")

In [19]:
import json

data_file = "final_project_tutorial_and_dev_data/cse476_final_project_dev_data.json"

with open(data_file, "r") as f:
    test_data = json.load(f)

MAX_QUESTIONS = 10   # start small

results = []

for i, item in enumerate(test_data[:MAX_QUESTIONS]):
    question = item["input"]

    print(f"Running question {i+1}/{MAX_QUESTIONS}")

    try:
        model_answer = solve(question)
    except Exception as e:
        print("Error:", e)
        model_answer = "ERROR"

    results.append({
        "question_number": i + 1,
        "input": question,
        "expected_output": item.get("output", ""),
        "model_output": model_answer,
        "domain": item.get("domain", "")
    })

print("Finished running selected questions.")

Running question 1/10
Running question 2/10
Running question 3/10
Running question 4/10
Running question 5/10
Running question 6/10
Running question 7/10
Running question 8/10
Running question 9/10
Running question 10/10
Finished running selected questions.


In [20]:
with open("my_outputs.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved to my_outputs.json")

Saved to my_outputs.json


### BONUS STUFF TO BE REMOVED


In [21]:
# %% Define three tests: input + expected
tests = [
    {
        "id": "math_inequality",
        "type": "numeric",  # grader will prefer numeric extraction
        "prompt": "Solve for the smallest integer n such that 3n + 5 > 26. Answer with just the integer.",
        "expected": "8",    # Because 3n > 21 => n > 7, smallest integer is 8
    },
    {
        "id": "commonsense_ice",
        "type": "text",
        "prompt": (
            "You place an ice cube in a glass of water and mark the water level. "
            "After the ice melts, does the water level rise, fall, or stay the same? "
            "Answer with exactly one of: 'rise', 'fall', 'stay the same'."
        ),
        "expected": "stay the same",
    },
    {
        "id": "logic_race",
        "type": "text",
        "prompt": (
            "In a race, you pass the person in second place. What position are you now in? "
            "Answer with a single word like 'first', 'second', 'third'."
        ),
        "expected": "second",
    },
]


In [22]:
# %% Simple normalization and evaluation helpers
def normalize_text(s: str) -> str:
    s = (s or "").strip().lower()
    # Remove surrounding punctuation and extra whitespace
    s = re.sub(r"[^\w\s\-']", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Map common synonyms used in these tests
    synonyms = {
        "unchanged": "stay the same",
        "no change": "stay the same",
        "same": "stay the same",
        "second place": "second",
        "2nd": "second",
        "first place": "first",
        "third place": "third",
    }
    return synonyms.get(s, s)

def extract_number(s: str):
    # Returns first number occurrence as string if found, else None
    if not s:
        return None
    m = re.search(r"[-+]?\d+(\.\d+)?", s)
    return m.group(0) if m else None

def grade(expected: str, got: str, kind: str) -> bool:
    if kind == "numeric":
        exp_num = extract_number(expected)
        got_num = extract_number(got)
        return (exp_num is not None) and (got_num == exp_num)
    else:
        return normalize_text(got) == normalize_text(expected)

def evaluate_tests(tests, model=MODEL):
    rows = []
    for t in tests:
        r = call_model_chat_completions(
            t["prompt"],
            system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
            model=model,
            temperature=0.0,
        )
        got = (r["text"] or "").strip()
        is_correct = grade(t["expected"], got, t["type"])
        rows.append({
            "id": t["id"],
            "expected": t["expected"],
            "got": got,
            "correct": is_correct,
            "status": r["status"],
            "error": r["error"],
        })
        # Tiny pacing to be polite to the API
        time.sleep(0.2)

    # Print a small report
    correct = sum(1 for x in rows if x["correct"])
    print(f"Score: {correct}/{len(rows)} correct")
    for x in rows:
        mark = "✅" if x["correct"] else "❌"
        print(f"{mark} {x['id']}: expected={x['expected']!r}, got={x['got']!r} (HTTP {x['status']})")
        if x["error"]:
            print("   error:", x["error"])
    return rows

results = evaluate_tests(tests)


Score: 2/3 correct
❌ math_inequality: expected='8', got='21' (HTTP 200)
✅ commonsense_ice: expected='stay the same', got='stay the same' (HTTP 200)
✅ logic_race: expected='second', got='second' (HTTP 200)


In [23]:
def self_evaluate(question, prediction, expected_answer, model=MODEL):
    """
    Use the model itself as a strict grader.
    Returns True if the model says the prediction matches the expected answer; else False.
    Falls back to a simple normalized string compare if the model's reply is malformed.
    """
    import re

    system = "You are a strict grader. Reply with exactly True or False. No punctuation. No explanation."
    prompt = f"""You are grading a question-answer pair.

Return exactly True if the PREDICTION would be accepted as correct for the EXPECTED_ANSWER.
Otherwise, return False.

QUESTION:
{question}

PREDICTION:
{prediction}

EXPECTED_ANSWER:
{expected_answer}

Answer with exactly: True or False
"""

    r = call_model_chat_completions(
        prompt,
        system=system,
        model=model,
        temperature=0.0,
    )

    reply = (r.get("text") or "").strip().lower()
    if reply.startswith("true"):
        return True
    if reply.startswith("false"):
        return False

    # Fallback: simple normalization-based equality
    norm = lambda s: re.sub(r"\s+", " ", (s or "").strip().lower())
    return norm(prediction) == norm(expected_answer)


In [24]:
def self_evaluate_tests(tests, model=MODEL, grader_model=None, sleep_sec=0.2, verbose=True):
    """
    Run the tests by querying the model for each prompt, then use LLM-as-a-judge
    (self_evaluate) to determine correctness.

    Args:
        tests: list of dicts with keys: id, prompt, expected (and optionally type)
        model: model used to generate predictions
        grader_model: model used to judge correctness (defaults to `model` if None)
        sleep_sec: small delay between calls to be polite to the API
        verbose: if True, print a summary line per test

    Returns:
        rows: list of dicts with fields:
              id, expected, got, correct, status, error
    """
    import time

    judge_model = grader_model or model
    rows = []

    for t in tests:
        # 1) Get model prediction
        r = call_model_chat_completions(
            t["prompt"],
            system="You are a careful solver. Reply ONLY with the final answer, nothing else.",
            model=model,
            temperature=0.0,
        )
        got = (r.get("text") or "").strip()

        # 2) LLM-as-a-judge: strict True/False
        is_correct = self_evaluate(
            question=t["prompt"],
            prediction=got,
            expected_answer=t["expected"],
            model=judge_model,
        )

        row = {
            "id": t.get("id", "<unnamed>"),
            "expected": t["expected"],
            "got": got,
            "correct": bool(is_correct),
            "status": r.get("status"),
            "error": r.get("error"),
        }
        rows.append(row)

        if verbose:
            mark = "✅" if is_correct else "❌"
            print(f"{mark} {row['id']}: expected={row['expected']!r}, got={row['got']!r} (HTTP {row['status']})")
            if row["error"]:
                print("   error:", row["error"])

        if sleep_sec:
            time.sleep(sleep_sec)

    return rows

# Example:
results_llm_judge = self_evaluate_tests(tests, verbose=True, model=MODEL, grader_model=MODEL)


❌ math_inequality: expected='8', got='21' (HTTP 200)
✅ commonsense_ice: expected='stay the same', got='stay the same' (HTTP 200)
✅ logic_race: expected='second', got='second' (HTTP 200)


In [25]:
# #Testing grader


# correct_count = 0

# for row in results:
#     is_correct = self_evaluate(
#         question=row["input"],
#         prediction=row["model_output"],
#         expected_answer=row["expected_output"]
#     )

#     row["correct"] = is_correct

#     if is_correct:
#         correct_count += 1

# accuracy = correct_count / len(results)

# print(f"Correct: {correct_count}/{len(results)}")
# print(f"Accuracy: {accuracy:.2%}")

In [26]:
correct_count = 0

for row in results:
    if "expected_output" not in row:
        continue  # skip — test data has no ground truth
    expected = row["expected_output"]
    got = row["model_output"]
    row["correct"] = grade(expected, got, "numeric")
    if row["correct"]:
        correct_count += 1

if correct_count > 0 or any("expected_output" in r for r in results):
    print(f"Correct: {correct_count}/{len(results)}")
    print(f"Accuracy: {correct_count / len(results):.2%}")
else:
    print("No ground truth available (test data). Run on dev data to evaluate.")

No ground truth available (test data). Run on dev data to evaluate.


In [27]:
with open("my_outputs_graded.json", "w") as f:
    json.dump(results, f, indent=4)

print("Saved graded results to my_outputs_graded.json")

Saved graded results to my_outputs_graded.json


In [ ]:
import json
from pathlib import Path

INPUT_PATH = "cse_476_final_project_test_data.json"
OUTPUT_PATH = "cse_476_final_project_answers.json"

with open(INPUT_PATH, encoding="utf-8") as f:
    questions = json.load(f)

answers = []
for i, item in enumerate(questions[:10]):
    print(f"Q {i+1}/10")
    try:
        answer = solve(item["input"])
    except Exception as e:
        print(f"  Error: {e}")
        answer = ""
    answers.append({"output": extract_final_answer(str(answer))})

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(answers, f, ensure_ascii=False, indent=2)

print(f"Saved {len(answers)} answers to {OUTPUT_PATH}")

Q 1/10
Q 2/10
Q 3/10
Q 4/10
Q 5/10
Q 6/10
Q 7/10
Q 8/10
Q 9/10
Q 10/10


UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 5: character maps to <undefined>